# Module C, sequence 1: the data objects (corrected version)

In [ ]:
import numpy as np
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data import Q_
from pymodaq_data.data import Axis, DataRaw, DataToExport

# linear axis: only scaling and offset are stored, use get_data() to get the array
wavelength = Axis('wavelength', 'nm', data=np.linspace(500, 560, 121), index=0)
signal = 3 * gauss1D(wavelength.get_data(), 528, 4) + 0.1
line = DataRaw('line', data=[signal], axes=[wavelength], labels=['photodiode'], units='V')

## 1. A linear axis does not store its array

In [ ]:
print(wavelength.data)  # None: a linear axis only keeps its scaling and offset
print(wavelength.get_data()[:3])

## 2. Navigation and signal axes

In [ ]:
print(line.dim, line.shape, line.nav_indexes, line.sig_indexes)
assert len(line.nav_indexes) == 0

## 3. Plot with the PyMoDAQ viewer (needs a display)

In [ ]:
%gui qt
line.plot('qt')

## 4. Slicing by values and by indices

In [ ]:
by_value = line.vsig[515.:540.]
by_index = line.isig[:10]
print(by_value.shape, by_value.axes[0].get_data()[[0, -1]])
print(by_index.shape)

## 5. Units: remove the background

In [ ]:
try:
    line - 0.1
except Exception as e:
    print(type(e).__name__, e)  # a bare number is not a voltage
no_bkg = line - Q_(0.1, 'V')
no_bkg.name = 'line_no_bkg'
print(no_bkg.units, no_bkg.data[0].max())

## 6. The axis in eV (on a copy)

In [ ]:
line_ev = line.deepcopy()
line_ev.axes[0] = line.axes[0].units_as('eV', inplace=False, context='spectroscopy')
print(line.axes[0].units, line.axes[0].get_data()[[0, -1]])
print(line_ev.axes[0].units, line_ev.axes[0].get_data()[[0, -1]])

## 7. Errors follow the slicing

In [ ]:
line.errors = [np.full(121, 0.05)]
print(line.vsig[515.:540.].errors[0].shape == line.vsig[515.:540.].data[0].shape)

## 8. Gather in a DataToExport

In [ ]:
mesure = DataToExport('mesure', data=[line, no_bkg])
print(mesure.get_data_from_name('line'))